# 第 05 课｜收益率的几种算法

> **数据身份**：固定随机种子生成的模拟教学数据，不是真实证券行情。  
> **环境**：`pkm-quant` · Python 3.11 · 默认离线运行。  
> **课程原始材料**：`05_youtube-8lEmreryblc/index.html`  
> **所属模块**：P1-M2 · 收益、时间价值与风险

## 本节量化问题

简单收益、对数收益和累计收益分别适合回答什么问题？

## 准确定义与策略报告位置

简单收益用于资产和组合财富变化；对数收益具有时间可加性；累计简单收益必须通过连乘得到。

- **核心关系**：简单收益 R_t=P_t/P_(t-1)-1；对数收益 r_t=ln(P_t)-ln(P_(t-1))；累计收益 ∏(1+R_t)-1。
- **写入报告**：收益口径、累计净值和年化规则
- **P2 衔接**：P2 第 32、34、36 课：收益矩阵、标签对齐和滚动计算。


## 运行环境与数据契约

先确认项目、数据目录和数据身份。后续联网数据源必须通过适配层进入，不能改变本 Notebook 的研究计算口径。

In [1]:
from pathlib import Path
import json
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

ROOT = Path.cwd()
if not (ROOT / "data" / "fixtures").exists():
    ROOT = ROOT.parent
DATA_DIR = ROOT / "data" / "fixtures"
OUTPUT_DIR = ROOT / "outputs"
OUTPUT_DIR.mkdir(exist_ok=True)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

RANDOM_SEED = 365
print("项目:", ROOT)
print("数据:", DATA_DIR)
print("数据身份: 固定随机种子生成的模拟教学数据，不是真实证券行情")


项目: /Users/mojiawen/Documents/学习资料归档/_暂存/personal_knowledge_manager/量化交易/实验/p1-quant-foundations
数据: /Users/mojiawen/Documents/学习资料归档/_暂存/personal_knowledge_manager/量化交易/实验/p1-quant-foundations/data/fixtures
数据身份: 固定随机种子生成的模拟教学数据，不是真实证券行情


## 用三个价格手算两种收益

简单收益先做价格比再减 1；对数收益先取自然对数再做差。小数组用于核对运算顺序。

In [2]:
prices_small = np.array([100.0, 105.0, 102.0])
simple_small = prices_small[1:] / prices_small[:-1] - 1.0
log_small = np.log(prices_small[1:]) - np.log(prices_small[:-1])

print("简单收益:", np.round(simple_small, 6))
print("对数收益:", np.round(log_small, 6))
assert np.isclose(np.prod(1.0 + simple_small) - 1.0, 102.0 / 100.0 - 1.0)

简单收益: [ 0.05     -0.028571]
对数收益: [ 0.04879  -0.028988]


## 在完整行情上核对累计收益

Pandas 保留日期和资产标签；累计简单收益用连乘，对数收益用累加后再指数还原，两者应一致。

In [3]:
daily = pd.read_csv(DATA_DIR / "daily_prices.csv", parse_dates=["date"])
close = daily.pivot(index="date", columns="symbol", values="adj_close").sort_index()
simple = close.pct_change(fill_method=None)
log_ret = np.log(close).diff()

cum_simple = (1.0 + simple).prod() - 1.0
cum_log = np.exp(log_ret.sum()) - 1.0
np.testing.assert_allclose(cum_simple, cum_log, rtol=1e-10)
display(pd.DataFrame({"simple_compounded": cum_simple, "log_reconstructed": cum_log}))

,simple_compounded,log_reconstructed
symbol,,
SIM_BOND,0.115428,0.115428
SIM_COMMODITY,-0.215188,-0.215188
SIM_DEFENSIVE,0.122820,0.122820
SIM_GROWTH,-0.033741,-0.033741
SIM_MARKET,-0.037347,-0.037347
SIM_VALUE,-0.091826,-0.091826


## 方法边界与阶段验收

对数收益不能按资产权重直接相加替代组合简单收益，年化也依赖频率和样本长度。

完成本课后应能：

1. 不看课程材料准确解释核心概念和公式符号；
2. 用小样本手工核对 Notebook 的关键结果；
3. 说明结果能支持什么判断、不能支持什么判断；
4. 指出它在策略报告中的位置，以及仍缺少的数据、成本或稳健性证据。
